# AI Stylist — Offline Evaluation Notebook

This notebook evaluates **logged recommendations against human relevance labels**. It does not train a model, call production APIs, connect to Supabase, or modify catalog data.

## Why this exists
- Measure recommendation quality with Precision@K, Recall@K, and NDCG@K.
- Check budget violations when both budget and product price are present.
- Compare runs only when they use the same query set and labeling rules.

## Dataset contract
Provide a CSV exported from a **sanitized, consent-appropriate offline evaluation set**. Do not include names, emails, auth tokens, private image URLs, or other personal data.

Required columns: `query_id`, `product_id`, `rank`, `relevant`.
- `query_id`: stable ID for one recommendation request; use a pseudonymous ID.
- `product_id`: catalog product ID or a pseudonymous ID.
- `rank`: 1-based position returned by the recommender.
- `relevant`: human-assessed binary label (`1` relevant, `0` not relevant).

Optional columns: `budget_max`, `product_price` (same currency and units). Rows without both values are excluded from the budget check.

A notebook metric is only as trustworthy as the relevance labels. Do not use model-generated labels as the sole ground truth.

In [ ]:
from pathlib import Path
import math
import pandas as pd

# Supports running from the repository root or from python/notebooks.
CANDIDATE_PATHS = [
    Path("python/notebooks/data/ai_stylist_eval.csv"),
    Path("data/ai_stylist_eval.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), CANDIDATE_PATHS[0])
K_VALUES = (3, 5, 10)

print(f"Evaluation CSV: {DATA_PATH}")
print("Expected columns: query_id, product_id, rank, relevant")

## Load and validate the evaluation data

The notebook intentionally fails fast on malformed input instead of silently dropping invalid rows. Create the CSV locally; do not commit private or production customer data.

In [ ]:
def load_evaluation_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Evaluation file not found: {path}. Create a sanitized CSV using the schema in the first cell."
        )

    df = pd.read_csv(path)
    required = {"query_id", "product_id", "rank", "relevant"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if df.empty:
        raise ValueError("Evaluation CSV has no rows.")
    if df[["query_id", "product_id", "rank", "relevant"]].isna().any().any():
        raise ValueError("Required columns contain null values.")

    df = df.copy()
    df["rank"] = pd.to_numeric(df["rank"], errors="raise")
    df["relevant"] = pd.to_numeric(df["relevant"], errors="raise")
    if (df["rank"] < 1).any() or (df["rank"] % 1 != 0).any():
        raise ValueError("rank must contain positive integers (1-based).")
    if not set(df["relevant"].unique()).issubset({0, 1}):
        raise ValueError("relevant must be binary: 0 or 1.")
    if df.duplicated(["query_id", "rank"]).any():
        raise ValueError("Each query_id must have at most one product at each rank.")

    return df.sort_values(["query_id", "rank"]).reset_index(drop=True)


# Run after placing the CSV at DATA_PATH.
# eval_df = load_evaluation_data(DATA_PATH)
# display(eval_df.head())

## Ranking metrics

- **Precision@K**: relevant items in the first K positions divided by K.
- **Recall@K**: relevant items in the first K positions divided by all relevant items labeled for that query. Queries with no positive labels are excluded from mean Recall@K and counted in the output metadata.
- **NDCG@K**: ranking-sensitive relevance score normalized by the ideal ordering.

Metrics are averaged per query, so a query with many candidate rows does not automatically dominate the result.

In [ ]:
def _dcg(relevances):
    return sum(float(rel) / math.log2(i + 2) for i, rel in enumerate(relevances))


def evaluate_query(group: pd.DataFrame, k: int) -> dict:
    ranked = group.sort_values("rank")
    top_k = ranked.head(k)
    rel_top_k = top_k["relevant"].astype(int).tolist()
    total_relevant = int(ranked["relevant"].sum())
    ideal = sorted(ranked["relevant"].astype(int).tolist(), reverse=True)[:k]
    ideal_dcg = _dcg(ideal)

    return {
        "query_id": group["query_id"].iloc[0],
        "precision_at_k": sum(rel_top_k) / k,
        "recall_at_k": (sum(rel_top_k) / total_relevant) if total_relevant else float("nan"),
        "ndcg_at_k": (_dcg(rel_top_k) / ideal_dcg) if ideal_dcg else 0.0,
        "labeled_relevant_total": total_relevant,
    }


def evaluate_recommendations(df: pd.DataFrame, k_values=K_VALUES) -> pd.DataFrame:
    results = []
    for k in k_values:
        if k < 1:
            raise ValueError("Every K must be >= 1.")
        for _, group in df.groupby("query_id", sort=False):
            row = evaluate_query(group, k)
            row["k"] = k
            results.append(row)

    per_query = pd.DataFrame(results)
    summary = (
        per_query.groupby("k", as_index=False)
        .agg(
            precision_at_k=("precision_at_k", "mean"),
            recall_at_k=("recall_at_k", "mean"),
            ndcg_at_k=("ndcg_at_k", "mean"),
            query_count=("query_id", "nunique"),
            queries_with_no_positive_labels=("labeled_relevant_total", lambda s: int((s == 0).sum())),
        )
    )
    return summary


# eval_df = load_evaluation_data(DATA_PATH)
# display(evaluate_recommendations(eval_df))

## Optional budget constraint check

This is a catalog/business-rule check, not a relevance metric. It runs only when both optional price columns exist. Ensure values use the same currency and units before interpreting the result.

In [ ]:
def budget_violation_summary(df: pd.DataFrame) -> pd.DataFrame:
    required = {"budget_max", "product_price"}
    if not required.issubset(df.columns):
        return pd.DataFrame([{
            "status": "not evaluated",
            "reason": "CSV must include both budget_max and product_price columns.",
        }])

    checked = df.dropna(subset=["budget_max", "product_price"]).copy()
    if checked.empty:
        return pd.DataFrame([{
            "status": "not evaluated",
            "reason": "No rows have both budget_max and product_price.",
        }])

    checked["budget_max"] = pd.to_numeric(checked["budget_max"], errors="raise")
    checked["product_price"] = pd.to_numeric(checked["product_price"], errors="raise")
    violations = checked["product_price"] > checked["budget_max"]
    return pd.DataFrame([{
        "status": "evaluated",
        "rows_checked": int(len(checked)),
        "budget_violations": int(violations.sum()),
        "budget_violation_rate": float(violations.mean()),
    }])


# display(budget_violation_summary(eval_df))

## Smoke test with toy data

The following tiny fixture checks the metric code only. **These values are synthetic and are not evidence of WEARO recommendation quality.**

In [ ]:
toy_df = pd.DataFrame([
    {"query_id": "toy-1", "product_id": "p-a", "rank": 1, "relevant": 1},
    {"query_id": "toy-1", "product_id": "p-b", "rank": 2, "relevant": 0},
    {"query_id": "toy-1", "product_id": "p-c", "rank": 3, "relevant": 1},
    {"query_id": "toy-2", "product_id": "p-d", "rank": 1, "relevant": 0},
    {"query_id": "toy-2", "product_id": "p-e", "rank": 2, "relevant": 1},
])

smoke_results = evaluate_recommendations(toy_df, k_values=(1, 3))
assert set(smoke_results["k"]) == {1, 3}
assert smoke_results["precision_at_k"].between(0, 1).all()
assert smoke_results["ndcg_at_k"].between(0, 1).all()
display(smoke_results)
print("Smoke test passed. This validates the metric implementation only, not model quality.")

## Decision rules before changing production weights

1. Use the same fixed query set and human-labeling rubric for every candidate run.
2. Compare current and candidate rankings on the same queries; report per-query results as well as aggregate means.
3. Review slices such as style, garment type, occasion, and budget when those labels are available and ethically collected.
4. Treat small differences cautiously; record sample size and uncertainty before changing weights.
5. Do not deploy a new scoring configuration solely because one aggregate metric improves. Check budget compliance, diversity, and human review too.

## Current limitations

- This notebook does not fetch recommendations from the application or infer labels automatically.
- It does not benchmark image-generation quality for Virtual Try-On; that requires a separate, controlled image-quality evaluation.
- No real quality score is reported until a valid, labeled CSV is supplied.